# Audit Your Classifier in 10 Minutes

**Goal.** Given an arbitrary binary classifier and a set of blockchain labels, find out whether the reported AUC is real behavioural signal or is leaking from the label-generating rule.

This notebook is the practitioner-oriented companion to the `onchain_audit` package. Run it top-to-bottom; wall-clock is under 10 minutes on a laptop. By the end you will have a Markdown audit report for your classifier with **PASS** or **FAIL** verdicts against three leakage indicators.

Prerequisites:
```
cd paper1_onchain_agent_id
pip install -e .
```

Structure:
1. Load the OnChainAgentID benchmark.
2. Train a reference classifier (GBM — you can replace with anything).
3. Run the three-step audit.
4. Interpret the verdict.
5. (Optional) Swap in your own classifier or your own dataset.

## 1. Setup & data loading

We load 1,147 addresses with 23 behavioural features and binary labels. The `AuditDataset` convenience loader also supplies the mining rules used to generate the labels and two purity tiers.

In [ ]:
import sys
from pathlib import Path

# Add the repo root to sys.path if running in-tree (not installed).
repo_root = Path.cwd().resolve()
while repo_root.name != 'paper1_onchain_agent_id' and repo_root.parent != repo_root:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

from onchain_audit import generate_audit_report
from onchain_audit.example import load_example_dataset

dataset = load_example_dataset()
print(f'Dataset:       {dataset.name}')
print(f'N addresses:   {len(dataset.labels)}')
print(f'N agents:      {int(dataset.labels.sum())}')
print(f'N humans:      {int((dataset.labels == 0).sum())}')
print(f'Features:      {dataset.features.shape[1]}')
print(f'Mining rules:  {list(dataset.mining_rules.columns)}')
print(f'Purity tiers:  {list(dataset.purity_tiers.keys())}')
print(f'Schemes:       {list(dataset.schemes.keys())}')

## 2. Train a reference classifier

Any sklearn-compatible binary classifier will do. GBM is the headline model in the paper; swap it for whatever you care about.

In [ ]:
# Your classifier — replace with whatever you actually want to audit.
my_classifier = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    min_samples_leaf=3,
    random_state=42,
)
my_classifier

## 3. Run the three-step audit

One call. Returns a Markdown string with every diagnostic table and risk flag.

In [ ]:
report = generate_audit_report(my_classifier, dataset)
print(report)

In [ ]:
# Render the Markdown report inline if you prefer:
from IPython.display import Markdown, display
display(Markdown(report))

## 4. Interpreting the verdict

The report's top-level verdict is a simple PASS/FAIL:

- **PASS** means no leakage indicator triggered.
- **FAIL** means at least one of Step 1, Step 2, or Step 3 flagged a problem.

### Step 1 — Label/feature overlap

High Pearson (|ρ| ≥ 0.6) or high Jaccard (≥ 0.5) between a rule you used to produce labels and a feature your model consumes is direct leakage. For this benchmark, Step 1 reliably flags `c1c4_c3_hour_entropy` vs `active_hour_entropy` — which is the exact bug identified in `FINDINGS_2026-04-08.md`.

### Step 2 — Purity-tier comparison

A large AUC drop (Δ ≥ 0.15 by default) between the full mined set and the hand-curated core means most of your model's lift is coming from the mining rule's statistical footprint, not from behavioural signal. On this benchmark, the drop is ~0.04 with RF — surprisingly mild, because the strict core (N=70) has too little data to discriminate fine-grained rule signals. On other datasets the drop can exceed 0.3.

### Step 3 — Cross-scheme transfer

Train on scheme A, test on scheme B, where the two schemes don't share a mining rule. A transfer gap ≥ 0.15 says your model is scheme-specific. On this benchmark it routinely exceeds 0.2 — LOPO-level evidence that agent-vs-human is not yet a cleanly separable concept with 23 Etherscan features.

## 5. Audit *your* classifier on *your* dataset

If you have your own binary-label dataset, the ceremony is the same — just build an `AuditDataset` from your tables.

Required pieces:

- `features` — `pd.DataFrame` indexed by address, numeric columns only.
- `labels` — `pd.Series` indexed by address, binary.
- `mining_rules` — `pd.DataFrame` indexed by address, one column per rule used to produce labels. Columns can be boolean, numeric, or string.
- `purity_tiers` — a dict mapping tier name to `(X, y)`, from lowest to highest purity.
- `schemes` — a dict mapping scheme name to `(X, y)` for two label schemes.

Example skeleton (replace with your own data):

In [ ]:
# Skeleton — substitute your own tables for the four placeholders below.
# Commented out so the notebook runs top-to-bottom without errors.

# from onchain_audit.report import AuditDataset
# my_features = pd.read_parquet('your_features.parquet')
# my_labels   = pd.read_csv('your_labels.csv', index_col='address')['label']
# my_rules    = pd.read_csv('your_mining_rules.csv', index_col='address')
#
# mask_curated = my_labels.index.isin(my_curated_addresses)
# tiers = {
#     'all_mined':   (my_features, my_labels),
#     'curated':     (my_features[mask_curated], my_labels[mask_curated]),
# }
#
# mask_a = my_labels.index.isin(scheme_a_addresses)
# mask_b = my_labels.index.isin(scheme_b_addresses)
# schemes = {
#     'scheme_A': (my_features[mask_a], my_labels[mask_a]),
#     'scheme_B': (my_features[mask_b], my_labels[mask_b]),
# }
#
# my_dataset = AuditDataset(
#     features=my_features,
#     labels=my_labels,
#     mining_rules=my_rules,
#     purity_tiers=tiers,
#     schemes=schemes,
#     name='my_project',
# )
# print(generate_audit_report(my_classifier, my_dataset))

## 6. What to do if you FAIL

The audit is diagnostic, not prescriptive. But in practice, FAIL verdicts usually call for one of:

1. **Remove the leaky feature.** If Step 1 flags a specific `(rule, feature)` pair, try re-training without that feature. Expect a large AUC drop — that drop *was* the leakage.
2. **Strengthen the labels.** Replace on-chain heuristic mining with at least one hand-curated or off-chain attestation per label (e.g., Flashbots relay data, Arkham entity label). See our `benchmark/splits/level4_strict_core.json` for the reference curated core.
3. **Scope the claim.** If Step 3 flags a cross-scheme gap, report AUC **within-platform only** and make the scope explicit. Our Paper 1's final claim is *within-scheme* at AUC ≈ 0.77; we explicitly avoid a cross-platform claim.

In all cases, include the audit report in your paper's appendix or supplementary material so reviewers can see that you checked.

---

## Appendix — saving the report


In [ ]:
out = Path('tutorial_audit_report.md')
out.write_text(report)
print(f'Wrote {out.resolve()}')